## 1. Setup
#### Load the API key and relevant Python libaries.

In [193]:
from collections import defaultdict
import json
import numpy as np
import pandas as pd
import os
import random

import io
from dotenv import dotenv_values
import openai
import os
from copy import deepcopy
import time
import ast
import re
import pprint
from openai import OpenAI

# LLM KEYS

In [194]:
# Get the first key from the uploaded dictionary
env_file_key = "key/env_c3_ai"

# Open the file and read its content
with open(env_file_key, 'r', encoding='utf-8') as file:
    env_content = file.read()

# Load the content into a variable
env_variables = dotenv_values(stream=io.StringIO(env_content))

api_key = env_variables['OPENAI_API_KEY']
# openai.api_key = api_key

client = OpenAI(
    # This is the default and can be omitted
    api_key=api_key,
)

# Functions

In [195]:
def parse_response(json_block):
    lines = json_block.split('\n')
    json_content = '\n'.join(lines[1:-1])
    return ast.literal_eval(json_content)

# Models

In [196]:
MODEL = "gpt-4.1-mini"
TEMPERATURE = 0

In [197]:
def chat_gpt(prompt, temperature=TEMPERATURE):
    response = client.chat.completions.create(
        model=MODEL,
        messages=prompt,
        temperature=temperature
    )
    return response.choices[0].message.content.strip()

# DATA PREPARATION

In [198]:
with open("data/aiid_dataset_25092025.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [199]:
# If data is a list of strings, parse them into dictionaries
if isinstance(data[0], str):
    data = [json.loads(item) for item in data]

rows = []
for incident in data:
    incident_id = incident.get("id")
    description = incident["incident"].get("description")
    full_description = incident["incident"].get("title")
    ai_system_description = incident["ai_card"]["model_card"].get("system_description")

    # safely handle data_inputs (can be None, list, or string)
    data_inputs = incident["ai_card"]["model_card"].get("data_inputs", [])
    if isinstance(data_inputs, list):
        input_modality = ", ".join(data_inputs)
    elif isinstance(data_inputs, str):
        input_modality = data_inputs
    else:
        input_modality = None

    ai_deployer = incident["incident"].get("ai_deployer")
    ai_developer = incident["incident"].get("ai_developer")

    harmed_parties_list = incident["incident"].get("harmed_parties", [])
    harmed_parties = ", ".join(harmed_parties_list) if isinstance(harmed_parties_list, list) else harmed_parties_list

    # Pick one random report text if available
    reports = incident["incident"].get("reports", [])
    report_text = None
    if isinstance(reports, list) and reports:
        chosen_report = random.choice(reports)
        report_text = chosen_report.get("text")

    rows.append({
        "incident_id": incident_id,
        "description": description,
        "full_description": full_description,
        "ai_system_description": ai_system_description,
        "input_modality": input_modality,
        "ai_deployer": ai_deployer,
        "ai_developer": ai_developer,
        "harmed_parties": harmed_parties,
        "random_report_text": report_text
    })

df = pd.DataFrame(rows)

In [200]:
df

,incident_id,description,full_description,ai_system_description,input_modality,ai_deployer,ai_developer,harmed_parties,random_report_text
0,1,YouTube’s content filtering and recommendation...,Google’s YouTube Kids App Presents Inappropria...,Content-recommendation and filtering algorithm,Youtube videos,[youtube],[youtube],children,"Update, Nov. 7, 2017: TODAY Parents is reshari..."
1,23,A self-driving public shuttle by Keolis North ...,Las Vegas Self-Driving Bus Involved in Accident,The autonomous shuttle bus was meant to traver...,"road data, lidar, video, odometer input, inert...","[navya, keolis-north-america]","[navya, keolis-north-america]","navya, keolis-north-america, bus-passengers",None of the eight passengers aboard the driver...
2,4,An Uber autonomous vehicle (AV) in autonomous ...,Uber AV Killed Pedestrian in Arizona,Autonomous driving system by Uber,"lidar, radar, video input, sensor data",[uber],[uber],"elaine-herzberg, pedestrians","RAFAELA VASQUEZ LIKED to work nights, alone, b..."
3,12,Researchers from Boston University and Microso...,Common Biases of Vector Embeddings,word2vec word embeddings trained on Google New...,"text, words, news articles","[microsoft-research, google, boston-university]","[microsoft-research, boston-university]","women, minority-groups",The blind application of machine learning runs...
4,5,Study on database reports of robotic surgery m...,Collection of Robotic Surgery Malfunctions,Surgery robot.,video input,[intuitive-surgical],"[hospitals, doctors]",patients,MIT Technology Review: According to a recent s...
...,...,...,...,...,...,...,...,...,...
1196,1208,Genians reported a phishing campaign by North ...,North Korea's Kimsuky Group Reportedly Uses AI...,None,None,[openai],"[kimsuky-group, velvet-chollima, group-0094, b...","south-korean-defense-personnel, government-of-...",North Korea's Kimsuky hackers use AI-generated...
1197,1209,13-year-old Juliana Peralta of Colorado report...,Lawsuit Alleges Character AI Chatbot Contribut...,None,None,[character.ai],[character.ai],"juliana-peralta, family-of-juliana-peralta, ch...",Juliana Peralta's mom got used to teachers cal...
1198,1210,Malicious versions of the popular Nx monorepo ...,Malicious Nx npm Packages Reportedly Weaponize...,None,None,"[anthropic, google, amazon]",[malicious-actors-compromising-nx's-cicd-pipel...,nx-users-and-organizations-installing-compromi...,LAS VEGAS --- While many business sectors are ...
1199,1211,"Stefanina's, a family-owned restaurant in Went...",Google AI Overviews Reportedly Misrepresented ...,None,None,[google],[google],"stefanina's-restaurant-(wentzville-missouri), ...","WENTZVILLE, Mo. (First Alert 4) -A popular loc..."


In [201]:
aiid_data = df

In [202]:
aiid_data.head(-5)

,incident_id,description,full_description,ai_system_description,input_modality,ai_deployer,ai_developer,harmed_parties,random_report_text
0,1,YouTube’s content filtering and recommendation...,Google’s YouTube Kids App Presents Inappropria...,Content-recommendation and filtering algorithm,Youtube videos,[youtube],[youtube],children,"Update, Nov. 7, 2017: TODAY Parents is reshari..."
1,23,A self-driving public shuttle by Keolis North ...,Las Vegas Self-Driving Bus Involved in Accident,The autonomous shuttle bus was meant to traver...,"road data, lidar, video, odometer input, inert...","[navya, keolis-north-america]","[navya, keolis-north-america]","navya, keolis-north-america, bus-passengers",None of the eight passengers aboard the driver...
2,4,An Uber autonomous vehicle (AV) in autonomous ...,Uber AV Killed Pedestrian in Arizona,Autonomous driving system by Uber,"lidar, radar, video input, sensor data",[uber],[uber],"elaine-herzberg, pedestrians","RAFAELA VASQUEZ LIKED to work nights, alone, b..."
3,12,Researchers from Boston University and Microso...,Common Biases of Vector Embeddings,word2vec word embeddings trained on Google New...,"text, words, news articles","[microsoft-research, google, boston-university]","[microsoft-research, boston-university]","women, minority-groups",The blind application of machine learning runs...
4,5,Study on database reports of robotic surgery m...,Collection of Robotic Surgery Malfunctions,Surgery robot.,video input,[intuitive-surgical],"[hospitals, doctors]",patients,MIT Technology Review: According to a recent s...
...,...,...,...,...,...,...,...,...,...
1191,1203,Prosecutors in Montana reportedly charged Shy ...,"Carter County, Montana Man Reportedly Charged ...",None,None,[unknown-generative-ai-developer],[shy-herbert-mccutchan],"unnamed-montana-child, unnamed-family-of-monta...","Missoula, MT ([KGVO-AM News](https://newstalkk..."
1192,1204,"On August 5, 2025, Greenwich, CT police found ...",ChatGPT Allegedly Reinforced Delusions Before ...,None,None,[openai],[erik-stein-soelberg],"stein-erik-soelberg, suzanne-eberson-adams","GREENWICH, Connecticut (WABC) -- The developer..."
1193,1205,Multiple AI systems allegedly spread false cla...,Multiple Generative AI Systems Reportedly Ampl...,None,None,"[xai, perplexity, google]","[xai, perplexity, google, pro-kremlin-actors]","general-public, grok-users, perplexity-users, ...",**What happened: **False claims surrounding th...
1194,1206,A 57-year-old woman in Bengaluru was allegedly...,Purported AI-Generated Deepfake of Spiritual L...,None,None,"[unknown-deepfake-technology-developer, unknow...","[scammers-impersonating-sadhguru, waleed-b-(sc...",unnamed-57-year-old-woman-in-bengaluru-targete...,A 57-year-old retired woman in Bengaluru has l...


# PROMPT

In [203]:
example_domains = """1. Biometric identification and categorization of natural persons
                2. Family
                3. Romantic relationships and friendships
                4. Health and Healthcare
                5. Well-being
                6. Human-Computer Interaction
                7. Finance and Investment
                8. Education and vocational training
                9. Employment, workers management and access to self-employment
                10. Essential private services and public services and benefits
                11. Recommender Systems and Personalization
                12. Social Media
                13. Sports and Recreation
                14. Arts and Entertainment
                15. Security and Cybersecurity
                16. Marketing and Advertising
                17. Agriculture and Farming
                18. Entrepreneurship
                19. Autonomous Robots and Robotics
                20. Innovation and Research
                21. Management and Operation of critical infrastructure
                22. Law enforcement
                23. Migration, Asylum and Border control management
                24. Democracy
                25. Media and Communication
                26. Accessibility and Inclusion
                27. Energy
                28. Military and Defense
                29. Administration of justice and democratic processes
                30. Government Services and Administration
                31. Diplomacy and Foreign Policy
                32. Food Safety and Regulation
                33. Crisis Management and Emergency Response
                34. Humanitarian Aid
                35. Transport and Logistics
                36. Urban Planning
                37. Counterterrorism
                38. Environment and Sustainability
                39. International Law Enforcement and Cooperation
                40. Climate Change Mitigation and Adaptation
                41. Gaming and interactive experiences
                42. Hobbies
                43. Smart home
                44. Social and Community Services
                45. Public and private transportation
                46. Interpersonal Communication"""

In [ ]:
MESSAGES = [
    {
        'role': 'system',
        'content': """
        As a Senior Technology Specialist, you specialize in the latest developments in Artificial Intelligence (AI) technology. You focus on Responsible AI development and use. As part of this, you investigate real-world AI incidents and try to understand both the original AI uses as well as specific, sometimes malicious, instantiations of those uses that caused the incidents. In this pivotal role, you are entrusted with reviewing and cataloguing the diverse applications, use cases, and incidents of AI technology across multiple domains.
        """
    },
    {
        'role': 'user',
        'content': """
        You will be provided input with the information from a database of real-world incidents, including:

        incidentID: "{}"
        description: "{}"   
        full_description: "{}"
        ai_system_description: "{}"
        input_modality: "{}"
        ai_deployer: "{}"
        ai_developer: "{}"
        harmed_parties: "{}"
        random_incident_report_text: "{}"

        Based on the input information, formulate the following outputs:

        I) The original intended AI use that has caused the incidents. You need to focus on the original AI use, as it might have been intended for potentially beneficial applications, even if it has been misused or its unintended applications resulted in the incident. 
        The definition of the use must contain specific details about how the technology is used by using action verbs that clearly describe the actions, activities, or processes of the uses. The level of specificity should be general and not on the very concrete instance. For each of these uses, you must output the following 6 elements each in less than 7 words:
        (1) Domain: The domain that represents the area or sector in which the AI system is intended to be used. We also provide an example list of possible domains you can use: example domains: "{}".
        (3) Purpose: The purpose or objective that is intended to be accomplished by using an AI system.
        (4) Capability: The capability of the AI system that enables the realization of its purpose and reflects the technological capability.
        (5) Space: The type of space in which the use took place. Can be one of: Online space; Publicly accessible space; Not publicly accessible space.
        (6) AI Deployer: The entity or individual in charge of deploying and managing the AI system, including individuals, organizations, corporations, public authorities, and agencies responsible for its operation and management. Even if the deployer of a general intended use is specified, such as the AI system provider, or if a specific person or entity misused the original use, do not name that entity directly but instead output the more general intended original deployer (e.g., "Social media company" instead of "Company X"). This is usually inferred from "ai_deployer" or "ai_developer" (e.g., if one of those fields is "amazon" and the incident description talks about "Amazon Alexa devices", then the AI Deployer is "Voice assistants providers").
        (7) AI Subject: The individual directly affected by the use of the AI system, experiencing its effects and consequences. They interact with or are impacted by the AI system's processes, decisions, or outcomes. If the general AI Deployer of the original intended AI use is specified, you should include it. If a specific person or entity were intentionally or unintentionally harmed but does not represent well the general deployer, output instead the general intended original deployer (e.g., "Social media users" instead of "John Doe"). This is usually inferred from "harmed_parties". Try to keep AI Subject different from AI Deployer; and to keep the naming consistency between the two in terms of terminology used.
        Ensure that each concept is specific and easy to understand for non-experts. Avoid duplicate purposes or objectives and use clear and precise language to describe the uses' concepts.

        For the "Capability", write it by combining action verbs in gerund form (i.e., ending with "ing"), inferences and data, entity or metric.

        (1) Action verbs clearly describe the actions, activities, or processes taken by the AI system, e.g., identify. Choose the most suitable action verb from the following list. If none can be assigned, propose a new verb, and mark it with an asterisk.
            (A) Estimating (e.g., Rating, Grading, Measuring, Assessing)
            (B) Forecasting (e.g., Predicting, Guessing, Speculating)
            (C) Comparing (e.g., Ranking, Ordering, Finding Best, Finding Cheapest, Recommending)
            (D) Detecting (e.g., Monitoring, Sensing, Noticing, Classifying, Discriminating)
            (E) Identifying (e.g., Recognizing, Discerning, Finding, Classifying, Perceiving)
            (F) Discovering (e.g., Extracting, Noticing, Organizing, Clustering, Grouping, Connecting, Revealing)
            (G) Generating (e.g., Making, Composing, Constructing, Creating, Authoring)
            (H) Acting (e.g., Doing, Executing, Playing, Going, Learning, Operating)
        (2) Inference clearly describes the output or conclusion drawn by the AI system based on the data it processes, e.g., crop yield, floods, trend, anomaly, wildfires, pattern, and probability
        (3) Data, Entity or Metric clearly describes the source, type, or nature of the data used by the AI system, e.g., from an optical camera, from an infrared camera, user input, sensor readings, transaction records, biometric data, environmental data, social media posts, geographical information, medical records, and financial metrics.


        For "Purpose", write it also in a gerund verb form (i.e., ending with "ing").

        Double-check that you are outputting realistic, i.e., plausible, meaningful, and useful uses.

        II) Describe the specific AI harms that occurred.
            Format each harm as one sentence, starting with the concrete harmed parties if specified, or else your identified/inferred harmed parties or subjects. Follow this by a verb in past tense specifying how they were harmed, and then "due to" and the specific reason (e.g., unintended use of the AI system, malfunctioning, misuse, technical capability risk or failure).
            For this part of the output, unlike the general AI use concepts, you must use the specific and concrete parties involved if they were named in the input (e.g., "Company X" instead of "Social media company" and "John Doe" instead of "social media users"). If the parties are not named in the input, use general terms without making up connections. For example, from the description "Researchers from Boston University and Microsoft Research, New England demonstrated gender bias in the most common techniques used to embed words for natural language processing (NLP)," you cannot conclude who the AI provider/deployer is because Boston University and Microsoft Research only demonstrated the bias but may not have built the system. In this case, you would output the harm as: "Women were discriminated against due to gender bias in word embeddings as demonstrated by researchers from Boston University and Microsoft Research."
            List as many distinct harms as you can clearly identify from the incident, but do not duplicate harms if the same or a similar harm affected multiple parties. Instead, include all those parties in a single harm description (e.g., "Pedestrians and motorists were endangered due to Uber's autonomous vehicles running red lights.").
            Be specific and name any parties involved—both the harmed ones and the AI deployers/providers—but only if they are listed in the input. End each harm description with a full stop (period).
            An example: if the input description is: "A Boeing 737 crashed into the sea, killing 189 people, after faulty sensor data caused an automated manuevering system to repeatedly push the plane's nose downward.", you can output one harm as: "Boeing 737 airplane passengers and crew were killed due to faulty sensor data causing an automated maneuvering system to repeatedly push the plane's nose downward.".

        Double-check that you are outputting all the harms that can be meaningfully identified, while not duplicating any across the subjects/harmed parties, and naming all the specific entities that you can correctly connect to the harm description. 
        
        *** Double-check your whole output and ensure that the AI use is described in general terms, while the harms are described in specific terms. ***
                
        Follow the example structure below for reporting the identified uses:
        Example 1:
            {{
                "IncidentID": 1,
                "Domain": "Biometric identification and categorization of natural persons",
                "Purpose": "Identifying individuals in crowded areas",
                "Capability": "Facial recognition from aerial footage",
                "Space": "Public Space",
                "AI Deployer": "Law enforcement agencies",
                "AI Subject": "Individuals in public spaces",
                "List of harms that occurred": ["Specific harmed parties" were "how harmed" due to "which reason -- relating to the specific AI system by specific provider/deployer/developer or its misuse", "-||-", ...]
            }}
        Example 2:
            {{
                "IncidentID": 2,
                "Domain": "Arts and Entertainment",
                "Purpose": "Creating altered images of people based on original photos",
                "Capability": "Generating manipulated images from original photos",
                "Space": "Online space",
                "AI Deployer": "Artists",
                "AI Subject": "People on photos",
                "List of harms that occurred": ["Specific harmed parties" were "how harmed" due to "which reason -- relating to the specific AI system by specific provider/deployer/developer or its misuse", "-||-", ...]
            }}
        Example 3:
            {{
                "IncidentID": 3,
                "Domain": "Arts and Entertainment",
                "Purpose": "Generating deepfakes resembling existing persons, objects, and places",
                "Capability": "Generating deepfake videos featuring faces",
                "Space": "Online space",
                "AI Deployer": "Deepfake application users",
                "AI Subject": "Celebrities",
                "List of harms that occurred": ["Specific harmed parties" were "how harmed" due to "which reason -- relating to the specific AI system by specific provider/deployer/developer or its misuse.", "-||-.", ...]
            }}
        Example 4:
            {{
                "IncidentID": 3,
                "Domain": "Transport and Logistics",
                "Purpose": "Maintaining aircraft stability during flight",
                "Capability": "Acting on sensor readings for stability",
                "Space": "Not publicly accessible space",
                "AI Deployer": "Aircraft manufacturers",
                "AI Subject": "Airplane passengers and crew",
                "List of harms that occurred": [
                    "Boeing 737 airplane passengers and crew were killed due to faulty sensor data causing an automated maneuvering system to repeatedly push the plane's nose downward."
                ]
            }}

        *** Ensure to output only the correctly formatted JSON and nothing else. ***
        """
    }
]


In [205]:
def format_prompt(MESSAGES, domains, row):
    messages = deepcopy(MESSAGES)
    incident_id = row["incident_id"]
    description = row["description"]
    full_description = row["full_description"]
    ai_system_description = row["ai_system_description"]
    input_modality = row["input_modality"]
    ai_deployer = row["ai_deployer"]
    ai_developer = row["ai_developer"]
    harmed_parties = row["harmed_parties"]
    random_report_text = row["random_report_text"]
    
    messages[1]['content'] = messages[1]['content'].format(incident_id, description, full_description, 
                                                           ai_system_description, input_modality,
                                                           ai_deployer, ai_developer,
                                                           harmed_parties, random_report_text, domains)
    return messages

# AIID Data

#### A. Run prompt manually on a sample

In [206]:
row1 = aiid_data.iloc[200]

In [207]:
row1["incident_id"]

np.int64(185)

In [208]:
messages = format_prompt(MESSAGES, example_domains, row1)
response = chat_gpt(messages)

In [209]:
print(response)

{
  "IncidentID": 185,
  "Domain": "Social Media",
  "Purpose": "Curating personalized content feeds",
  "Capability": "Comparing user engagement signals for recommending",
  "Space": "Online space",
  "AI User": "Social media companies",
  "AI Subject": "Social media users",
  "List of harms that occurred": [
    "TikTok users and new users were exposed to false and misleading content about the Russia-Ukraine war due to TikTok's algorithm pushing disinformation within minutes of app sign-up.",
    "TikTok users and new users were misinformed due to the algorithm failing to distinguish between reliable and false sources or provide fact-checks and warnings.",
    "TikTok users and new users were exposed to Kremlin propaganda and false narratives due to the algorithm blending real and false content without transparency or source trustworthiness indicators."
  ]
}


#### B. Run batch processing

In [ ]:
# Creating an array of json tasks
tasks = []

for index, row in aiid_data.iterrows():
    useI = row["incident_id"]
    
    
    # adapt the prompt for useI
    messagesInc = format_prompt(MESSAGES, example_domains, row)

    print(index)
    
    task = {
        "custom_id": f"task-{int(useI)}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            # This is what you would have in your Chat Completions API call
            "model": MODEL,
            "temperature": TEMPERATURE,
            "response_format": { 
                "type": "json_object"
            },
            "messages": messagesInc,
        }
    }
    
    tasks.append(task)

0 incident_id                                                              1
description              YouTube’s content filtering and recommendation...
full_description         The content filtering system for YouTube's chi...
ai_system_description    "A content filtering system incorporating mach...
input_modality                                                      Videos
ai_deployer                                                    ["youtube"]
ai_developer                                                   ["youtube"]
harmed_parties                                                    children
harm_distribution                                                      Age
Name: 0, dtype: object
1 incident_id                                                              2
description              Twenty-four Amazon workers in New Jersey were ...
full_description         On December 5, 2018, a robot punctured a can o...
ai_system_description                                                  Na

In [142]:
tasks

[{'custom_id': 'task-1',
  'method': 'POST',
  'url': '/v1/chat/completions',
  'body': {'model': 'gpt-4.1-mini',
   'temperature': 0,
   'response_format': {'type': 'json_object'},
   'messages': [{'role': 'system',
     'content': '\n        As a Senior Technology Specialist, you specialize in the latest developments in Artificial Intelligence (AI) technology. You focus on Responsible AI development and use. As part of this, you investigate real-world AI incidents and try to understand both the original AI uses as well as specific, sometimes malicious, instantiations of those uses that caused the incidents. In this pivotal role, you are entrusted with reviewing and cataloguing the diverse applications, use cases, and incidents of AI technology across multiple domains.\n        '},
    {'role': 'user',
     'content': '\n        You will be provided input with the information from a database of real-world incidents, including:\n\n        incidentID: "1"\n        description: "YouTube’

# Save Batch File

In [143]:
# Creating the file

file_name = "batches/batch_general_uses_specific_harms_AIID.jsonl"

with open(file_name, 'w') as file:
    for obj in tasks:
        file.write(json.dumps(obj) + '\n')

# Uploading the File

In [144]:
batch_file = client.files.create(
  file=open(file_name, "rb"),
  purpose="batch"
)

In [145]:
print(batch_file)

FileObject(id='file-R9Dy2RapfsJ6wz12Br3bws', bytes=92152, created_at=1759314604, filename='batch_general_uses_specific_harms_AIID.jsonl', object='file', purpose='batch', status='processed', expires_at=1761906604, status_details=None)


# Creating the batch job

In [146]:
batch_job = client.batches.create(
  input_file_id=batch_file.id,
  endpoint="/v1/chat/completions",
  completion_window="24h",
  metadata={"project": "telco-risks", "prompt_version": "v01102025"}
)

# Checking the batch status

In [148]:
batch_job = client.batches.retrieve(batch_job.id)
#print("Batch job:", batch_job)
print("Completed batches:", batch_job.request_counts.completed, "out of", batch_job.request_counts.total)
# Wait until completed=N of the records we send there

Completed batches: 6 out of 6


# Retrieving the results

In [149]:
result_file_id = batch_job.output_file_id
result = client.files.content(result_file_id).content

In [150]:
result_file_name = "results/batch_v1_output.jsonl"

with open(result_file_name, 'wb') as file:
    file.write(result)

# Loading data from saved file and transforming into JSON

In [152]:
result_file_name = "results/batch_v1_output.jsonl"

# Loading data from saved file
results = []
with open(result_file_name, 'r') as file:
    for line in file:
        # Parsing the JSON string into a dict and appending to the list of results
        json_object = json.loads(line.strip())
        message_content = json_object['response']['body']['choices'][0]['message']['content']
        results.append(message_content)

In [153]:
for res in results:
    print (res)
    break

{
  "IncidentID": 1,
  "Domain": "Recommender Systems and Personalization",
  "Purpose": "Filtering and recommending appropriate videos",
  "Capability": "Detecting inappropriate content and recommending videos",
  "Space": "Online space",
  "AI User": "Social media company",
  "AI Subject": "Children",
  "List of harms that occurred": [
    "Children were exposed to disturbing and inappropriate videos due to YouTube's content filtering system failing to screen out unsuitable material.",
    "Children were exposed to videos containing sex, drugs, violence, profanity, and conspiracy theories due to YouTube's recommendation algorithm promoting inappropriate content.",
    "Children were exposed to thousands of videos resembling popular cartoons but containing age-inappropriate content due to failures in YouTube's algorithmic and human review filters.",
    "Children were exposed to harmful content despite YouTube's additional filters like 'restricted mode' failing to block all inappropri

In [154]:
results

['{\n  "IncidentID": 1,\n  "Domain": "Recommender Systems and Personalization",\n  "Purpose": "Filtering and recommending appropriate videos",\n  "Capability": "Detecting inappropriate content and recommending videos",\n  "Space": "Online space",\n  "AI User": "Social media company",\n  "AI Subject": "Children",\n  "List of harms that occurred": [\n    "Children were exposed to disturbing and inappropriate videos due to YouTube\'s content filtering system failing to screen out unsuitable material.",\n    "Children were exposed to videos containing sex, drugs, violence, profanity, and conspiracy theories due to YouTube\'s recommendation algorithm promoting inappropriate content.",\n    "Children were exposed to thousands of videos resembling popular cartoons but containing age-inappropriate content due to failures in YouTube\'s algorithmic and human review filters.",\n    "Children were exposed to harmful content despite YouTube\'s additional filters like \'restricted mode\' failing to 

# SAVE

In [155]:
# Convert each JSON string to a Python dict
parsed_results = [json.loads(r) for r in results]

In [ ]:
# Use json.dump to write the list to a file in JSON format
with open('data/extreacted_intended_uses.json', 'w') as json_file:
    json.dump(parsed_results, json_file, indent=4, ensure_ascii=False)  # 4 spaces of indentation